In [1]:
import pandas as pd
import json

Data Structure

In [3]:
#Read three main datasets
transactions = pd.read_csv('./data/transactions_data.csv')
cards = pd.read_csv('./data/cards_data.csv')
users = pd.read_csv('./data/users_data.csv')

In [4]:
#Read two json files
with open('./data/mcc_codes.json') as f:
    mcc_codes = json.load(f)

with open('./data/train_fraud_labels.json') as f:
    fraud_labels = json.load(f)

In [5]:
#Look basic info
for name, df in [('transactions', transactions), ('cards', cards), ('users', users)]:
    print(f"\n{'='*40}")
    print(f"TABLE: {name}")
    print(f"Shape: {df.shape}")
    print(df.dtypes)
    print(df.head(2))


TABLE: transactions
Shape: (13305915, 12)
id                  int64
date               object
client_id           int64
card_id             int64
amount             object
use_chip           object
merchant_id         int64
merchant_city      object
merchant_state     object
zip               float64
mcc                 int64
errors             object
dtype: object
        id                 date  client_id  card_id   amount  \
0  7475327  2010-01-01 00:01:00       1556     2972  $-77.00   
1  7475328  2010-01-01 00:02:00        561     4575   $14.57   

            use_chip  merchant_id merchant_city merchant_state      zip   mcc  \
0  Swipe Transaction        59935        Beulah             ND  58523.0  5499   
1  Swipe Transaction        67570    Bettendorf             IA  52722.0  5311   

  errors  
0    NaN  
1    NaN  

TABLE: cards
Shape: (6146, 13)
id                        int64
client_id                 int64
card_brand               object
card_type                object
c

JSON files

In [6]:
#Structure of mcc_codes
print("mcc_codes sample:")
sample_mcc = dict(list(mcc_codes.items())[:3])
print(json.dumps(sample_mcc, indent=2))

mcc_codes sample:
{
  "5812": "Eating Places and Restaurants",
  "5541": "Service Stations",
  "7996": "Amusement Parks, Carnivals, Circuses"
}


In [7]:
#Structure of fraud_labels
print("\nfraud_labels type:", type(fraud_labels))
if isinstance(fraud_labels, dict):
    keys = list(fraud_labels.keys())[:3]
    print("sample keys:", keys)
    print("sample values:", {k: fraud_labels[k] for k in keys})
elif isinstance(fraud_labels, list):
    print("sample:", fraud_labels[:3])

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



Build DuckDB

In [10]:
import duckdb

In [11]:
#Build Database
con = duckdb.connect('./data/fraud_analytics.db')

In [12]:
# ── 1. transactions ──────────────────────────────────────────
transactions = pd.read_csv('./data/transactions_data.csv')

# amount has $ symbol，clean to float
transactions['amount'] = transactions['amount'].str.replace('$', '', regex=False).astype(float)

con.execute("""
    CREATE OR REPLACE TABLE transactions AS
    SELECT * FROM transactions
""")
print(f"✓ transactions: {con.execute('SELECT COUNT(*) FROM transactions').fetchone()[0]:,} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ transactions: 13,305,915 rows


In [13]:
# ── 2. cards ─────────────────────────────────────────────────
cards = pd.read_csv('./data/cards_data.csv')
cards['credit_limit'] = cards['credit_limit'].str.replace('$', '', regex=False).str.replace(',', '').astype(float)

# credit_limit has $ symbol，clean to float
con.execute("""
    CREATE OR REPLACE TABLE cards AS
    SELECT * FROM cards
""")
print(f"✓ cards: {con.execute('SELECT COUNT(*) FROM cards').fetchone()[0]:,} rows")

✓ cards: 6,146 rows


In [14]:
# ── 3. users ──────────────────────────────────────────────────
users = pd.read_csv('./data/users_data.csv')
for col in ['per_capita_income', 'yearly_income', 'total_debt']:
    users[col] = users[col].str.replace('$', '', regex=False).str.replace(',', '').astype(float)

# 'per_capita_income', 'yearly_income', 'total_debt' have $ symbol，clean to float
con.execute("""
    CREATE OR REPLACE TABLE users AS
    SELECT * FROM users
""")
print(f"✓ users: {con.execute('SELECT COUNT(*) FROM users').fetchone()[0]:,} rows")

✓ users: 2,000 rows


In [15]:
# ── 4. mcc_codes ──────────────────────────────────────────────
with open('./data/mcc_codes.json') as f:
    mcc_codes = json.load(f)

mcc_df = pd.DataFrame([
    {'mcc': int(k), 'description': v}
    for k, v in mcc_codes.items()
])

con.execute("""
    CREATE OR REPLACE TABLE mcc_codes AS
    SELECT * FROM mcc_df
""")
print(f"✓ mcc_codes: {con.execute('SELECT COUNT(*) FROM mcc_codes').fetchone()[0]:,} rows")

✓ mcc_codes: 109 rows


In [17]:
# ── 5. fraud_labels ───────────────────────────────────────────
with open('./data/train_fraud_labels.json') as f:
    fraud_labels = json.load(f)

labels = fraud_labels.get('target', fraud_labels)
fraud_df = pd.DataFrame([
    {'transaction_id': int(k), 'is_fraud': 1 if str(v).strip().lower() == 'yes' else 0}
    for k, v in labels.items()
])

con.execute("""
    CREATE OR REPLACE TABLE fraud_labels AS
    SELECT * FROM fraud_df
""")
print(f"✓ fraud_labels: {con.execute('SELECT COUNT(*) FROM fraud_labels').fetchone()[0]:,} rows")

print("\n🎉 Database is Built！")

✓ fraud_labels: 8,914,963 rows

🎉 Database is Built！


Sanity Check: If the tables have a normal relationship

In [18]:
con = duckdb.connect('./data/fraud_analytics.db')

In [19]:
#1. Number of rows in each table
print("=== Number of Rows ===")
for table in ['transactions', 'cards', 'users', 'mcc_codes', 'fraud_labels']:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:20s}: {count:>12,}")

=== Number of Rows ===
transactions        :   13,305,915
cards               :        6,146
users               :        2,000
mcc_codes           :          109
fraud_labels        :    8,914,963


In [20]:
#2. Multiple Table JOIN Test: Complete info for a transaction
print("\n=== Mutiple Table Join Test（1 Transaction）===")
result = con.execute("""
    SELECT
        t.id            AS transaction_id,
        t.date,
        t.amount,
        t.use_chip,
        u.id            AS user_id,
        u.gender,
        u.credit_score,
        c.card_brand,
        c.card_type,
        m.description   AS merchant_category,
        fl.is_fraud
    FROM transactions t
    JOIN users u        ON t.client_id = u.id
    JOIN cards c        ON t.card_id   = c.id
    JOIN mcc_codes m    ON t.mcc       = m.mcc
    JOIN fraud_labels fl ON t.id       = fl.transaction_id
    LIMIT 1
""").df()

print(result.to_string(index=False))


=== Mutiple Table Join Test（1 Transaction）===
 transaction_id                date  amount          use_chip  user_id gender  credit_score card_brand card_type merchant_category  is_fraud
        7769051 2010-03-16 10:29:00    8.05 Swipe Transaction     1445   Male           841 Mastercard     Debit   Discount Stores         0


In [21]:
#3. Fraud Ratio
print("\n=== Fraud Ratio ===")
con.execute("""
    SELECT
        is_fraud,
        COUNT(*)                              AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM fraud_labels
    GROUP BY is_fraud
""").df().pipe(print)


=== Fraud Ratio ===
   is_fraud    count    pct
0         0  8901631  99.85
1         1    13332   0.15


Data is normal and relationships also match.